# Corrected-data benchmark - DCRNN

The processed array the benchmark used is out of date order: its first 48 rows
are 2023, placed before 2013, so every training split contained data from after
its own test period. It also drops 49 weekly reports and stores one week twice.
`docs/ARRAY_AUDIT.md` has the evidence.

This kernel re-runs **DCRNN** on three versions of the case series with **one
training loop**, so the data is the only thing that changes:

| dataset | what it is |
|---|---|
| `original` | the array as the benchmark used it (459 rows) |
| `reordered` | the same rows in true date order, duplicate removed (451 weeks) |
| `rebuilt` | every source report on a regular weekly grid (559 weeks) |

Frozen protocol unchanged: 3 origins x 3 seeds, window 3 -> horizon 3,
pooled RMSE. The artifact is located by report, not by row 395, and windows
whose input or target touches a week with no source report are dropped (nothing is filled). Arms: `base` and `spatial`.

`original` should reproduce the earlier numbers; if it does not, stop and compare.

## 1. Environment

In [ ]:
import subprocess, sys, os, json, time, hashlib, re
from pathlib import Path

REPO = "https://github.com/MLOpenSourceOpenScience/disease_modeling_MLOS2.git"
COMMIT = "45f1c0878f002407633ed1237638734faa9ceb2b"
BLOB_NPY = "f7cfa6ec31a4058584fe256a1d6de6800e72a5b1"
BLOB_ADJ = "f3a3cb7f43998850410b0a494f16f331c3830a84"
SEGMENTS = [0.6, 0.7, 0.8, 0.9, 1.0]     # the authors' __main__ segment list

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
# Build outside /kaggle/working: anything left there becomes kernel output, and a
# 40 MB clone plus a venv makes `kaggle kernels output` unusably slow.
SCRATCH = Path("/tmp/repro") if Path("/tmp").exists() else WORK
SCRATCH.mkdir(parents=True, exist_ok=True)
SRC = SCRATCH / "mlos2"
VENV = SCRATCH / "venv311"
PY311 = VENV / "bin" / "python"

os.environ["MPLBACKEND"] = "Agg"          # their plotting helpers call plt.show()


def sh(*args, check=True, quiet=False, **kw):
    """Run a command, echoing it, and abort on a non-zero exit."""
    if not quiet:
        print("$", " ".join(str(a) for a in args))
    r = subprocess.run([str(a) for a in args], text=True, capture_output=True, **kw)
    if r.stdout.strip() and not quiet:
        print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        if check:
            raise SystemExit("command failed: " + " ".join(str(a) for a in args))
    return r


print("kernel python:", sys.version.split()[0])

In [ ]:
# Kaggle runs Python 3.12; torch 2.1.2 has no cp312 wheel. Fetch a standalone
# 3.11 with uv rather than bumping the authors' pinned torch.
sh(sys.executable, "-m", "pip", "install", "-q", "uv")

UV = [sys.executable, "-m", "uv"]
sh(*UV, "python", "install", "3.11")
sh(*UV, "venv", "--python", "3.11", str(VENV))

PIP = [*UV, "pip", "install", "-q", "--python", str(PY311)]

# Exactly the versions in the authors' requirements.txt.
sh(*PIP, "torch==2.1.2", "--index-url", "https://download.pytorch.org/whl/cpu")
sh(*PIP, "torch_scatter", "torch_sparse", "-f",
   "https://data.pyg.org/whl/torch-2.1.2+cpu.html")

# Deviation 2: the authors pin torch_geometric==2.5.3, but PGT 0.54.0 imports
# torch_geometric.utils.to_dense_adj, which PyG removed in 2.4. 2.4.0 is the
# newest release where all five architectures import.
sh(*PIP, "torch_geometric==2.4.0", "numpy~=1.26.2", "pandas~=2.2.0",
   "scikit_learn==1.4.0", "statsmodels==0.14.1", "decorator==4.4.2",
   "cython", "matplotlib", "tqdm")

# Deviation 1: PGT's own pandas<=1.3.5 pin contradicts the authors'
# pandas~=2.2.0 and has no Python 3.11 wheel.
sh(*PIP, "--no-deps", "torch_geometric_temporal==0.54.0")

In [ ]:
probe = sh(str(PY311), "-c", """
import json, torch, torch_geometric, pandas, numpy
from torch_geometric_temporal import A3TGCN, ASTGCN, AAGCN
from torch_geometric_temporal.nn.recurrent import DCRNN
from torch_geometric_temporal.signal import StaticGraphTemporalSignal, temporal_signal_split
print(json.dumps({
    "python": ".".join(map(str, __import__("sys").version_info[:3])),
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "pandas": pandas.__version__,
    "numpy": numpy.__version__,
}))
""", quiet=True)

versions = json.loads(probe.stdout.strip().splitlines()[-1])
print(json.dumps(versions, indent=2))
assert versions["python"].startswith("3.11"), versions["python"]
assert versions["torch"].startswith("2.1.2"), versions["torch"]
assert versions["torch_geometric"] == "2.4.0", versions["torch_geometric"]
print("all five architectures import OK under Python 3.11")

## 2. Clone

In [ ]:
ARCH = "DCRNN"
PROJECT = "https://github.com/rathishTharusha/dengue-forecasting-gnn.git"
BRANCH = "exp/seir-gnn"
PROJ = SCRATCH / "project"
if not PROJ.exists():
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, PROJECT, str(PROJ))
RUNNER = PROJ / "analysis" / "_build" / "run_corrected_benchmark.py"
assert RUNNER.exists(), f"Runner not found at {RUNNER}"
assert (PROJ / "data" / "corrected" / "rebuilt_cases.npy").exists(), "corrected data missing"
print("Cloned", BRANCH)

## 3. Run all three datasets

In [ ]:
started = time.time()
proc = subprocess.Popen(
    [str(PY311), "-u", str(RUNNER), "--arch", ARCH, "--datasets", "original", "reordered", "rebuilt",
     "--seeds", "3", "--out-dir", str(WORK)],
    cwd=str(PROJ), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, f"runner failed with exit code {proc.returncode}"
print(f"\nfinished in {(time.time() - started) / 60:.1f} min")

## 4. Summary

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

rows = []
for f in sorted(Path(WORK).glob("corrected_*.json")):
    rows.extend(json.loads(f.read_text(encoding="utf-8")))
df = pd.DataFrame(rows).drop_duplicates(["dataset", "arch", "increment", "origin", "seed"])

floor = (df[df.arch == "persistence"].groupby(["dataset", "origin"])["RMSE_clean"].mean())
model = df[df.arch != "persistence"]
table = (model.groupby(["dataset", "arch", "increment", "origin"])["RMSE_clean"].mean()
         .reset_index())
table["floor"] = [floor[(d, o)] for d, o in zip(table.dataset, table.origin)]
table["vs_floor"] = table.RMSE_clean - table.floor
summary = (table.groupby(["dataset", "arch", "increment"], sort=False)
           .agg(RMSE_clean=("RMSE_clean", "mean"), floor=("floor", "mean"),
                vs_floor=("vs_floor", "mean"), wins=("vs_floor", lambda s: int((s < 0).sum()))))
pd.set_option("display.width", 160)
print(summary.round(3).to_string())